In [ ]:
%pip install -q openai python-dotenv pillow matplotlib

In [ ]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
import matplotlib.pyplot as plt

load_dotenv(override=True, dotenv_path="../.env.local")
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is missing. Add it to ../.env.local")

client = OpenAI(api_key=api_key)
print(f"API Key loaded: {api_key[:4]}...{api_key[-4:]}")

In [ ]:
# Change this path if you want to analyze a different cabinet image
image_path = Path("data/2026-02-16 13.34.42.jpg")

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

img = Image.open(image_path)
plt.figure(figsize=(6, 8))
plt.imshow(img)
plt.axis("off")
plt.title(f"Cabinet Image: {image_path.name}")
plt.show()

In [ ]:
with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

suffix = image_path.suffix.lower()
mime_type = "image/png" if suffix == ".png" else "image/jpeg"

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "system",
            "content": "You are a cabinet quality inspector. Identify visible defects and explain clearly."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Inspect this cabinet image and return: 1) defect type, 2) defect location, 3) short defect description, 4) severity (Low/Medium/High), 5) suggested action."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{mime_type};base64,{image_b64}"
                    }
                }
            ]
        }
    ]
)

print("Cabinet Defect Analysis:\n")
print(response.choices[0].message.content)

In [ ]:
# Batch analysis for all cabinet images
import json

image_paths = sorted(Path('data').glob('2026-02-16 *.jpg'))
if not image_paths:
    image_paths = sorted(Path('data').glob('*.jpg')) + sorted(Path('data').glob('*.jpeg')) + sorted(Path('data').glob('*.png'))

if not image_paths:
    raise FileNotFoundError('No images found in data/')

def extract_json(text: str) -> dict:
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return {"raw_response": text.strip()}
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return {"raw_response": text.strip()}

rows = []
for path in image_paths:
    with open(path, 'rb') as f:
        image_b64 = base64.b64encode(f.read()).decode('utf-8')

    mime_type = 'image/png' if path.suffix.lower() == '.png' else 'image/jpeg'

    resp = client.chat.completions.create(
        model='gpt-5-nano',
        messages=[
            {
                'role': 'system',
                'content': 'You are a cabinet quality inspector. Return only valid JSON.'
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': (
                            'Inspect this cabinet image and respond in JSON with keys: '
                            'defect_type, defect_location, defect_description, severity, suggested_action.'
                        )
                    },
                    {
                        'type': 'image_url',
                        'image_url': {'url': f'data:{mime_type};base64,{image_b64}'}
                    }
                ]
            }
        ]
    )

    content = resp.choices[0].message.content or ''
    data = extract_json(content)

    rows.append({
        'image': path.name,
        'defect_type': data.get('defect_type', 'N/A'),
        'defect_location': data.get('defect_location', 'N/A'),
        'severity': data.get('severity', 'N/A'),
        'suggested_action': data.get('suggested_action', 'N/A')
    })

headers = ['image', 'defect_type', 'defect_location', 'severity', 'suggested_action']
width = {h: max(len(h), *(len(str(r[h])) for r in rows)) for h in headers}

line = ' | '.join(h.ljust(width[h]) for h in headers)
sep = '-+-'.join('-' * width[h] for h in headers)
print(line)
print(sep)
for r in rows:
    print(' | '.join(str(r[h]).ljust(width[h]) for h in headers))
